In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import joblib

# Set MLflow Tracking URI
mlflow.set_tracking_uri("http://localhost:5000")  # Change if running remotely

# Enable MLflow experiment
mlflow.set_experiment("fruit_classification")

# Load dataset
df = pd.read_csv("data/raw/fruit_data_with_colors.txt", sep="\t")

# Feature selection
features = ["mass", "width", "height", "color_score"]
target = "fruit_label"

X = df[features]
y = df[target]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save the scaler as an artifact
joblib.dump(scaler, "models/scaler.joblib")

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Define parameter grid
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10]
}

# Perform Grid Search with Cross-Validation
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

# Train model with MLflow tracking
with mlflow.start_run():
    grid_search.fit(X_train, y_train)

    # Get the best model and parameters
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    # Make predictions
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # Log best hyperparameters
    mlflow.log_params(best_params)

    # Log accuracy metric
    mlflow.log_metric("accuracy", accuracy)

    # Save and log model
    joblib.dump(best_model, "models/model.joblib")
    mlflow.sklearn.log_model(best_model, "model")

    # Log scaler as an artifact
    mlflow.log_artifact("models/scaler.joblib")

    print(f"Best Model: {best_params}")
    print(f"Model trained and logged with accuracy: {accuracy}")


MlflowException: API request to http://localhost:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=fruit_classification (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000018960A41CD0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))